# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert metadata to a Python dict for pretty printing
metadata_json = dataset.metadata.to_json()
print("Dataset name: {}\nDescription: {}".format(metadata_json.get('name'), metadata_json.get('description')))


## 2. Data Overview
Review available record sets, fields, and their `@id` references.

The Croissant schema organizes tabular records as `RecordSet` entities, each with its own `@id`. Each `RecordSet` consists of `Field` entities, which are mapped to columns and also have their own `@id`s.

Let's enumerate all available record sets and their fields, referencing entities by their `@id`.

In [ ]:
# List all record sets and their fields using @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets:")

for rs in record_sets:
    print("\nRecordSet @id: {}".format(rs['@id']))
    print("RecordSet name: {}".format(rs.get('name', '[no name]')))
    print("Fields:")
    for field in rs.get('fields', []):
        print(f"  Field @id: {field['@id']} | name: {field.get('name', '[no name]')} | dataType: {field.get('dataType')}")

### Example Record Preview
Preview the first record from each `RecordSet` by referencing the record set `@id`.


In [ ]:
# Display example record from each RecordSet
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample from RecordSet {rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i == 0:
                break
    except Exception as e:
        print(f"Could not load records from {rs_id}: {e}")

## 3. Data Extraction
Load tabular data from the available record sets into DataFrames for analysis. 

All entities are referenced using their `@id`. 
Let's load records for each record set.

In [ ]:
# Gather record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet {rs_id} with columns:")
            print(df.columns.tolist())
            print(df.head())
        else:
            print(f"No records found for RecordSet {rs_id}")
    except Exception as e:
        print(f"Error loading RecordSet {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric field criteria, normalizing values, and grouping data by key field attributes.

All processing references entities by `@id`.

In [ ]:
# Choose an example RecordSet and numeric field for analysis
# For illustration, pick the first RecordSet and scan it for a numeric field

chosen_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(chosen_record_set_id)

# Find numeric field @id in the first record set
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    if rs['@id'] == chosen_record_set_id:
        for field in rs.get('fields', []):
            if field.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field_id = field['@id']
            # Use a categorical field as group (such as anatomical location, if present)
            if field.get('dataType') == 'schema:Text':
                group_field_id = field['@id']
        break

# Fallback for demonstration if not found
if df is not None and numeric_field_id is None:
    # Try to pick a numeric column name (assume @id used as DataFrame column)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if df is not None and group_field_id is None:
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}")

# EDA: Filtering, normalization, grouping
if df is not None and numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized field '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("Unable to perform EDA: No numeric field found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the grouping field (if available).

This example uses matplotlib for basic visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of numeric field ({numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Access metadata and records from the FAIR^2 Colorectal Cancer dataset using `mlcroissant`
- Reference all entities by their `@id` for traceability
- Extract data from various record sets into DataFrames
- Perform basic EDA: filtering, normalization, and grouping
- Visualize numeric distributions and groupings

You can extend your analysis by referencing further fields (`@id`), exploring more groupings, and deploying richer statistical models as suited for clinical and molecular exploration.